In [ ]:
import os, json
import pandas as pd
from f1_eval import run_f1_loro, f1_report, type_recall, run_detail
from scope import apply_scope, scope_tag

EXCLUDE_INITIAL, EXCLUDE_TRANSIENT = True, False
TAG = scope_tag(EXCLUDE_INITIAL, EXCLUDE_TRANSIENT)
OUT = f'./outputs/f1_{TAG}'
os.makedirs(OUT, exist_ok=True)

# 06에서 추천된 조합 사용 (직접 지정하려면 CONFIG를 덮어쓰기)
best = json.load(open(f'./outputs/loro_{TAG}/best_config.json', encoding='utf-8'))
CONFIG = {p: (v['combo'], v['model']) for p, v in best.items()}
# CONFIG = {'cn7': ('T0+state', 'iforest'), 'rg3': ('C5_event+state', 'logit')}   # 이전 조합과 비교할 때
print(CONFIG)

RESULT = {}
for p, (combo, model) in CONFIG.items():
    d = apply_scope(pd.read_csv(f'./base/{p}_features.csv', parse_dates=['TimeStamp']),
                    EXCLUDE_INITIAL, EXCLUDE_TRANSIENT)
    print(f'\n{p.upper()} — {combo} + {model} ...')
    oof = run_f1_loro(d, combo, model)
    oof.to_csv(f'{OUT}/{p}_f1_oof.csv', index=False, encoding='utf-8-sig')
    RESULT[p] = oof

    reps = [f1_report(oof, label='전체')]
    if p == 'cn7':
        reps.append(f1_report(oof, eval_mask=oof['Reason'] != '가스', label='미성형 계열(가스 제외)'))
    rep = pd.concat(reps, ignore_index=True)
    rep.to_csv(f'{OUT}/{p}_f1_report.csv', index=False, encoding='utf-8-sig')
    display(rep.round(3))

    for main in ['pred_Bthr', 'pred_Bfrac']:
        print(f'--- {main} ---')
        display(type_recall(oof, main))
        display(run_detail(oof, main))
json.dump({p: list(v) for p, v in CONFIG.items()}, open(f'{OUT}/config.json', 'w', encoding='utf-8'),
          ensure_ascii=False)

CN7 — T0+state + iforest ...


,평가셋,방법,F1,Precision,Recall,TP,FP,FN,판정 불량 수
0,전체,pred_Bthr,0.429,0.750,0.300,6,2,14,8
1,전체,pred_Bfrac,0.279,0.261,0.300,6,17,14,23
2,전체,pred_A990,0.074,0.040,0.500,10,242,10,252
3,전체,pred_A995,0.137,0.082,0.400,8,89,12,97
4,전체,C_top5%,0.131,0.078,0.400,8,94,12,102
5,전체,C_top10%,0.082,0.045,0.450,9,191,11,200
6,전체,D_사후최적(참고),0.552,0.889,0.400,8,1,12,9
7,이벤트형(가스 제외),pred_Bthr,0.706,0.750,0.667,6,2,3,8
8,이벤트형(가스 제외),pred_Bfrac,0.375,0.261,0.667,6,17,3,23
9,이벤트형(가스 제외),pred_A990,0.062,0.032,0.889,8,242,1,250


선택된 비율(fold별): {'cn7_R1': 0.01, 'cn7_R2': 0.01, 'cn7_R3': 0.01, 'cn7_R4': 0.01, 'cn7_R5': 0.01, 'cn7_R6': 0.01}


,탐지,전체
Reason,,
가스,0,11
가스+미성형,1,2
미성형,1,2
초기허용불량,4,5


,샷,불량,탐지(TP),오탐(FP)
run_id,,,,
cn7_R1,188,14,2,0
cn7_R2,418,0,0,5
cn7_R3,314,5,4,0
cn7_R4,189,0,0,2
cn7_R5,407,0,0,5
cn7_R6,463,1,0,5


RG3 — C5_event+state + logit ...


,평가셋,방법,F1,Precision,Recall,TP,FP,FN,판정 불량 수
0,전체,pred_Bthr,0.155,0.100,0.344,11,99,21,110
1,전체,pred_Bfrac,0.094,0.075,0.125,4,49,28,53
2,전체,C_top5%,0.091,0.088,0.094,3,31,29,34
3,전체,C_top10%,0.082,0.061,0.125,4,62,28,66
4,전체,D_사후최적(참고),0.215,0.212,0.219,7,26,25,33


선택된 비율(fold별): {'rg3_R1': 0.2, 'rg3_R2': 0.15, 'rg3_R3': 0.03, 'rg3_R4': 0.2, 'rg3_R5': 0.02, 'rg3_R6': 0.075}


,탐지,전체
Reason,,
가스,3,22
미성형,1,10


,샷,불량,탐지(TP),오탐(FP)
run_id,,,,
rg3_R1,41,0,0,9
rg3_R2,103,5,0,16
rg3_R3,197,10,0,6
rg3_R4,74,4,2,13
rg3_R5,176,6,0,4
rg3_R6,36,7,2,1


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from features import FoldFeaturizer, combo_columns, needs_fold
from scope import apply_scope, scope_tag

TAG = scope_tag(True, False)
OUT = f'./outputs/f1_{TAG}'
COMBO = json.load(open(f'{OUT}/config.json', encoding='utf-8'))['cn7'][0]
METHOD = 'pred_Bthr'
COLS = combo_columns(COMBO)

d = apply_scope(pd.read_csv('./base/cn7_features.csv', parse_dates=['TimeStamp']), True, False)
oof = pd.read_csv(f'{OUT}/cn7_f1_oof.csv')
print('조합:', COMBO, '| 변수', len(COLS), '개')

_av = {f.name for f in font_manager.fontManager.ttflist}
for _f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR']:
    if _f in _av:
        mpl.rcParams['font.family'] = _f; break
mpl.rcParams.update({'axes.unicode_minus': False, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e6e5df', 'grid.linewidth': 0.6,
                     'axes.edgecolor': '#b5b4ad', 'axes.labelcolor': '#52514e',
                     'xtick.color': '#6f6e69', 'ytick.color': '#6f6e69'})
INK, MUTED = '#0b0b0b', '#898781'
TYPE_ORDER = ['미성형', '가스+미성형', '가스']
TYPE_COL = {'정상': '#c3c2b7', '미성형': '#4a3aa7', '가스+미성형': '#e87ba4', '가스': '#e34948'}
DIV = LinearSegmentedColormap.from_list('div', ['#184f95', '#6da7ec', '#f0efec', '#ef8a88', '#b52a2a'])
short = lambda c: (c.replace('_raw', '').replace('Mold_Temperature_', 'Mold_').replace('Hopper_Temperature', 'Hopper')
                    .replace('Barrel_Temperature_', 'Barrel_').replace('Max_', '').replace('Average_', 'Avg_'))


# ① 방법별 유형 탐지 수
def detect_table(oof, methods):
    pos, rows = oof[oof['y'] == 1], []
    for m in methods:
        r = {'방법': m}
        for t in TYPE_ORDER:
            g = pos[pos['Reason'] == t]
            r[t] = f"{int(g[m].sum())}/{len(g)}"
        ex = pos[pos['Reason'] != '가스']
        r['미성형 계열'] = f"{int(ex[m].sum())}/{len(ex)}"
        r['전체'] = f"{int(pos[m].sum())}/{len(pos)}"
        r['오탐(FP)'] = int(((oof[m] == 1) & (oof['y'] == 0)).sum())
        rows.append(r)
    return pd.DataFrame(rows)

display(detect_table(oof, [c for c in ['pred_Bthr', 'pred_Bfrac', 'pred_A995'] if c in oof]))


# 변수별 정상 대비 z (테스트 Run마다 다른 Run 양품 기준 = 모델이 본 기준)
def fold_z(d, cols, combo, clip=15):
    parts = []
    for r in d['run_id'].unique():
        tr, te = d[d['run_id'] != r], d[d['run_id'] == r]
        if needs_fold(combo):
            ff = FoldFeaturizer().fit(tr)
            tr, te = ff.transform(tr), ff.transform(te)
        base = tr[(tr['PassOrFail'] == 0) & (tr['shots_since_stop'] > 5)][cols]
        med = base.median()
        scale = ((base - med).abs().median() * 1.4826)
        scale = scale.where(scale > 0, base.std()).replace(0, 1).fillna(1)
        z = ((te[cols] - med) / scale).clip(-clip, clip)
        for c in ['run_id', 'shot_seq_in_run', 'PassOrFail', 'Reason']:
            z[c] = te[c].values
        parts.append(z)
    return pd.concat(parts, ignore_index=True)

Z = fold_z(d, COLS, COMBO).merge(
    oof[['run_id', 'shot_seq', 'score', METHOD, 'B_thr']],
    left_on=['run_id', 'shot_seq_in_run'], right_on=['run_id', 'shot_seq'], how='left')
print(Z.shape, '| 병합 안 된 행:', Z['score'].isna().sum())

In [ ]:
# ② Run별 위험 점수 타임라인
runs = Z['run_id'].unique()
fig, axes = plt.subplots(len(runs), 1, figsize=(14, 2.3 * len(runs)))
for ax, r in zip(axes, runs):
    g = Z[Z['run_id'] == r].sort_values('shot_seq_in_run')
    ax.plot(g['shot_seq_in_run'], g['score'], color=MUTED, lw=1, zorder=1)
    thr = g['B_thr'].iloc[0]
    if np.isfinite(thr):
        ax.axhline(thr, color=INK, ls='--', lw=1, zorder=1)
        ax.text(g['shot_seq_in_run'].max(), thr, ' 임계값', va='center', fontsize=8, color='#52514e')
    for t in TYPE_ORDER:
        h = g[g['Reason'] == t]
        ax.scatter(h['shot_seq_in_run'], h['score'], s=50, color=TYPE_COL[t],
                   edgecolor='white', linewidth=1.5, zorder=3, label=t)
    tp = g[(g[METHOD] == 1) & (g['PassOrFail'] == 1)]
    fp = g[(g[METHOD] == 1) & (g['PassOrFail'] == 0)]
    ax.scatter(tp['shot_seq_in_run'], tp['score'], s=160, facecolor='none', edgecolor=INK,
               linewidth=1.3, zorder=4, label='탐지됨(TP)')
    ax.scatter(fp['shot_seq_in_run'], fp['score'], s=70, marker='x', color=INK, linewidth=1.3,
               zorder=4, label='오탐(FP)')
    ax.set_title(f"{r} — 불량 {int(g['PassOrFail'].sum())}, 탐지 {len(tp)}, 오탐 {len(fp)}",
                 loc='left', fontsize=10, color=INK)
    ax.set_ylabel('위험 점수')
axes[-1].set_xlabel('Run 내 샷 순번')
h, l = [], []
for ax in axes:
    for hh, ll in zip(*ax.get_legend_handles_labels()):
        if ll not in l:
            h.append(hh); l.append(ll)
fig.legend(h, l, loc='upper right', ncol=6, frameon=False, fontsize=9)
fig.suptitle(f'CN7 — Run별 위험 점수 ({COMBO}, 초기허용 제외)', x=0.01, ha='left', color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.97)); fig.savefig(f'{OUT}/cn7_score_timeline.png', dpi=130); plt.show()

In [ ]:
# ③ 불량 샷 + 오탐 샷 × 변수 히트맵
D = Z[(Z['PassOrFail'] == 1) | (Z[METHOD] == 1)].copy()
D['유형'] = np.where(D['PassOrFail'] == 1, D['Reason'], '오탐')
order = TYPE_ORDER + ['오탐']
D['유형'] = pd.Categorical(D['유형'], order, ordered=True)
D = D.sort_values(['유형', 'run_id', 'shot_seq_in_run'])

M = D[COLS].fillna(0).values
fig, ax = plt.subplots(figsize=(15, 0.36 * len(D) + 2.5))
im = ax.imshow(M, cmap=DIV, vmin=-10, vmax=10, aspect='auto')
ylab = [f"{r} #{s}  {t}{'  ●탐지' if (p == 1 and y == 1) else ''}"
        for r, s, t, p, y in zip(D['run_id'], D['shot_seq_in_run'], D['유형'], D[METHOD], D['PassOrFail'])]
ax.set_yticks(range(len(D))); ax.set_yticklabels(ylab, fontsize=8)
ax.set_xticks(range(len(COLS))); ax.set_xticklabels([short(c) for c in COLS], rotation=60, ha='right', fontsize=8)
for b in np.flatnonzero(D['유형'].values[1:] != D['유형'].values[:-1]):
    ax.axhline(b + 0.5, color=INK, lw=1)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.5, label='정상 대비 z (다른 Run 양품 기준, ±10에서 색 고정)')
ax.set_title('불량·오탐 샷은 어떤 변수에서 정상과 달랐나', loc='left', color=INK)
fig.tight_layout(); fig.savefig(f'{OUT}/cn7_defect_heatmap.png', dpi=130); plt.show()

In [ ]:
# ④ 유형별 프로파일 + 유형마다 가장 크게 벗어난 변수 5개
grp = Z.copy()
grp['유형'] = np.where(grp['PassOrFail'] == 1, grp['Reason'], '정상')
prof = grp.groupby('유형')[COLS].median().reindex(['정상'] + TYPE_ORDER)

fig, ax = plt.subplots(figsize=(15, 3.2))
im = ax.imshow(prof.values, cmap=DIV, vmin=-8, vmax=8, aspect='auto')
ax.set_yticks(range(len(prof))); ax.set_yticklabels([f"{t} (n={int((grp['유형'] == t).sum())})" for t in prof.index])
ax.set_xticks(range(len(COLS))); ax.set_xticklabels([short(c) for c in COLS], rotation=60, ha='right', fontsize=8)
for i in range(prof.shape[0]):
    for j in range(prof.shape[1]):
        v = prof.values[i, j]
        if np.isfinite(v) and abs(v) >= 2:
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=7,
                    color='white' if abs(v) > 6 else INK)
ax.grid(False); fig.colorbar(im, ax=ax, shrink=0.8, label='중앙값 z')
ax.set_title('유형별로 어떤 변수가 어느 방향으로 벗어났나 (|z|≥2만 숫자 표시)', loc='left', color=INK)
fig.tight_layout(); fig.savefig(f'{OUT}/cn7_type_profile.png', dpi=130); plt.show()

top5 = {t: prof.loc[t].abs().sort_values(ascending=False).head(5) for t in TYPE_ORDER}
display(pd.DataFrame({t: [f"{short(c)} ({prof.loc[t, c]:+.1f})" for c in s.index] for t, s in top5.items()}))

In [ ]:
# ⑤ 차이가 큰 변수 10개: 정상 vs 3가지 유형
rank = prof.loc[TYPE_ORDER].abs().max().sort_values(ascending=False)
TOP = rank.head(10).index.tolist()
rng = np.random.default_rng(0)
cats = ['정상'] + TYPE_ORDER

fig, axes = plt.subplots(2, 5, figsize=(17, 7), sharey=True)
for ax, c in zip(axes.flat, TOP):
    for i, t in enumerate(cats):
        v = grp.loc[grp['유형'] == t, c].dropna()
        if t == '정상' and len(v) > 500:
            v = v.sample(500, random_state=0)
        x = i + rng.uniform(-0.18, 0.18, len(v))
        ax.scatter(x, v, s=10 if t == '정상' else 36, color=TYPE_COL[t],
                   edgecolor='white' if t != '정상' else 'none', linewidth=1, alpha=0.6 if t == '정상' else 1)
    lo, hi = grp.loc[grp['유형'] == '정상', c].quantile([0.05, 0.95])
    ax.axhspan(lo, hi, color='#e6e5df', zorder=0)
    ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats, rotation=30, fontsize=8)
    ax.set_title(short(c), fontsize=10, color=INK)
    ax.set_ylim(-16, 16)
axes[0, 0].set_ylabel('정상 대비 z'); axes[1, 0].set_ylabel('정상 대비 z')
fig.suptitle('정상 vs 불량 유형 — 차이가 큰 변수 10개 (회색 띠 = 정상 중앙 90%, ±15에서 자름)',
             x=0.01, ha='left', color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.95)); fig.savefig(f'{OUT}/cn7_top_features.png', dpi=130); plt.show()